In [1]:
#importing packages and env-var

from pyspark.sql.session import SparkSession
import pyspark.sql.functions as f

import os
from dotenv import load_dotenv

env = load_dotenv(dotenv_path='/mnt/d/linux/project/vars/.env')

#env vars for postgresql connection
mssqlsource = os.getenv('MSSQLSOURCE')
mssqluser = os.getenv('MSSQL_SPARK_USER')
mssqlpwd = os.getenv('MSSQL_SPARK_PWD')
mssqldriver = os.getenv('MSSQL_DRIVER')
sqlserverjar = os.getenv('SQLSERVER_JAR')
mssql_dwdest = os.getenv('MSSQL_CONN')

In [2]:
#create session
spark = (( SparkSession
            .builder
            .appName('orders')
            .config('spark.jars', sqlserverjar)
            .config('spark.driver.extraClassPath', sqlserverjar)
            .config('spark.executor.extraClassPath', sqlserverjar)
            .getOrCreate()
        ))

your 131072x1 screen size is bogus. expect trouble
24/04/15 12:53:06 WARN Utils: Your hostname, DOUGPC resolves to a loopback address: 127.0.1.1; using 172.25.197.48 instead (on interface eth0)
24/04/15 12:53:06 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
24/04/15 12:53:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/04/15 12:53:09 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
tmp_orders = (( spark
                .read
                .format('jdbc')
                .option('url', mssqlsource)
                .option('driver', mssqldriver)
                .option('user', mssqluser)
                .option('password', mssqlpwd)
                .option('query', 
                        """
                            select
                            
                                orderid,
                                customerid,
                                salespersonpersonid,
                                customerpurchaseordernumber,
                                pickedbypersonid,
                                backorderorderid,
                                contactpersonid,
                                orderdate,
                                expecteddeliverydate,
                                pickingcompletedwhen as processedtime
                                
                            from sales.orders
                            
                        """)
            )).load()

In [4]:
tmp_orderline = (( spark
                    .read
                    .format('jdbc')
                    .option('url', mssqlsource)
                    .option('driver', mssqldriver)
                    .option('user', mssqluser)
                    .option('password', mssqlpwd)
                    .option('query', 
                            
                            """
                                select

                                    orderid,
                                    orderlineid,
                                    stockitemid,
                                    packagetypeid,
                                    quantity,
                                    pickedquantity,
                                    unitprice,
                                    taxrate,
                                    pickingcompletedwhen
                                    
                                from sales.orderlines
                            
                            """)
                )).load()

In [5]:
join_order = (( tmp_orders
                .join(tmp_orderline, 'orderid', 'left')
            ))

In [7]:
#ajust hierarchy columns
#removing nulls
#changing datetime to date

orders = (( join_order
            .select('orderid',
                    'orderlineid',
                    'customerpurchaseordernumber',
                    'stockitemid',
                    'customerid',
                    'salespersonpersonid',
                    'packagetypeid',
                    'pickedbypersonid',
                    'backorderorderid',
                    'contactpersonid',
                    'orderdate',
                    'expecteddeliverydate',
                    'pickingcompletedwhen',
                    'processedtime',
                    'quantity',
                    'pickedquantity',
                    'unitprice',
                    'taxrate'
                    )
            
            #cleaning and adjust some datatype columns.
            .withColumn('pickingcompletedwhen', f.expr('cast(pickingcompletedwhen as timestamp)'))
                                                .withColumnRenamed('pickingcompletedwhen', 'orderstart')
                                                
            .withColumn('processedtime', f.expr('cast(processedtime as timestamp)'))
                                        .withColumnRenamed('processedtime', 'orderfinalized')
            
            .withColumn('pickedbypersonid', f.expr('coalesce(pickedbypersonid, 0)'))
            
            .withColumn('orderdate', f.expr('cast(orderdate as date)'))
            
            .withColumn('expecteddeliverydate', f.expr('cast(expecteddeliverydate as date)'))
                                        .withColumnRenamed('expecteddeliverydate', 'deliverydate')
            
            .withColumn('backorderorderid', f.expr('coalesce(backorderorderid, 0)'))
            
            .withColumn('unitprice', f.expr('cast(unitprice as decimal(12, 2))'))
            
            .withColumn('taxrate', f.expr('cast(taxrate as decimal(5, 2))'))
            
            .withColumnRenamed('customerpurchaseordernumber', 'customerpurchaseorder')

        ))

In [8]:
((
    orders
        .write
        .format('jdbc')
            .option('url', mssql_dwdest)
            .option('driver', mssqldriver)
            .option('user', mssqluser)
            .option('password', mssqlpwd)
            .option('dbtable', 'fact.sales')
        .save(mode= 'overwrite')
))

24/04/15 12:53:37 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
